# Dataset maestro

In [ ]:
import pandas as pd

## Carga de datos y preparación

In [2]:
df_results = pd.read_csv('../data/raw/results.csv')
df_rankings = pd.read_csv('../data/raw/fifa_ranking-2026-04-01.csv')
df_names = pd.read_csv('../data/raw/former_names.csv')

In [3]:
df_results['date'] = pd.to_datetime(df_results['date'])
df_rankings['rank_date'] = pd.to_datetime(df_rankings['rank_date'])

In [4]:
# Diccionario de mapeo de nombres antiguos a nuevos
name_dict = dict(zip(df_names['former'], df_names['current']))

In [5]:
df_results.replace({'home_team': name_dict, 'away_team': name_dict}, inplace=True)
df_rankings.replace({'country_full': name_dict}, inplace=True)

,Unnamed: 0,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,0,140.0,Brunei Darussalam,BRU,2.0,0.00,140,AFC,1992-12-31
1,1,33.0,Portugal,POR,38.0,0.00,33,UEFA,1992-12-31
2,2,32.0,Zambia,ZAM,38.0,0.00,32,CAF,1992-12-31
3,3,31.0,Greece,GRE,38.0,0.00,31,UEFA,1992-12-31
4,4,30.0,Algeria,ALG,39.0,0.00,30,CAF,1992-12-31
...,...,...,...,...,...,...,...,...,...
70189,70189,229.0,Brunei Darussalam,BRU,633.0,875.78,-41,AFC,2026-04-01
70190,70190,231.0,US Virgin Islands,VIR,627.0,776.60,-24,CONCACAF,2026-04-01
70191,70191,232.0,Cook Islands,COK,622.0,877.53,-46,OFC,2026-04-01
70192,70192,233.0,Macau,MAC,589.0,865.29,-40,AFC,2026-04-01


## Merge

In [6]:
df_results = df_results.sort_values('date')
df_rankings = df_rankings.sort_values('rank_date')

In [7]:
df_master = pd.merge_asof(
    df_results,
    df_rankings[['rank_date', 'country_full', 'rank', 'total_points']],
    left_on='date', right_on='rank_date',
    left_by='home_team', right_by='country_full'
).rename(columns={'rank': 'rank_home', 'total_points': 'points_home'}).drop(['rank_date', 'country_full'], axis=1)

In [8]:
df_master = pd.merge_asof(
    df_master,
    df_rankings[['rank_date', 'country_full', 'rank', 'total_points']],
    left_on='date', right_on='rank_date',
    left_by='away_team', right_by='country_full'
).rename(columns={'rank': 'rank_away', 'total_points': 'points_away'}).drop(['rank_date', 'country_full'], axis=1)

In [9]:
df_master.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank_home,points_home,rank_away,points_away
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,NaN,NaN,NaN,NaN
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,NaN,NaN,NaN,NaN
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,NaN,NaN,NaN,NaN
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,NaN,NaN,NaN,NaN
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,NaN,NaN,NaN,NaN


## Variables de interés

In [10]:
df_master = df_master[df_master['date'] >= '1993-01-01'].copy()

In [11]:
df_master['points_diff'] = df_master['points_home'] - df_master['points_away']

In [12]:
df_master['score_diff'] = df_master['home_score'] - df_master['away_score']

In [13]:
df_master.dropna(subset=['points_home', 'points_away'], inplace=True)

In [14]:
df_master.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank_home,points_home,rank_away,points_away,points_diff,score_diff
18704,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,Libreville,Gabon,True,39.0,34.0,69.0,22.0,12.0,0.0
18705,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,Libreville,Gabon,False,55.0,27.0,97.0,11.0,16.0,0.0
18706,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,Kuwait City,Kuwait,False,71.0,21.0,161.0,0.0,21.0,2.0
18707,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,Libreville,Gabon,True,97.0,11.0,69.0,22.0,-11.0,1.0
18708,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,Libreville,Gabon,False,55.0,27.0,39.0,34.0,-7.0,-1.0


In [15]:
# Cantidad de registros finales
print(f"Cantidad de registros en el dataset final: {len(df_master)}")

Cantidad de registros en el dataset final: 23991


In [16]:
df_future_mundial = df_master[df_master['home_score'].isna()].copy()

In [17]:
df_history = df_master[df_master['home_score'].notna()].copy()

In [18]:
df_history.dropna(subset=['points_home', 'points_away'], inplace=True)

### Correlación entre las variables creadas

In [19]:
matriz_corr = df_history[['points_diff', 'score_diff']].corr(method='pearson')
r_value = matriz_corr.iloc[0, 1]

In [20]:
print(f"Coeficiente de correlación: {r_value:.4f}")

Coeficiente de correlación: 0.4488


## Exportación de datasets

In [21]:
df_history.to_csv('../data/processed/master_entrenamiento.csv', index=False)

In [22]:
df_future_mundial.to_csv('../data/processed/fixture_predict_2026.csv', index=False)